# Microfinance Fraud Detection

## Data Preprocessing

### Objective

This notebook prepares the cleaned dataset for machine learning by performing train-test splitting, preprocessing, encoding categorical features, scaling numerical features, and building a reusable preprocessing pipeline.

In [55]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

import joblib

In [56]:
df = pd.read_csv("C:\\Users\\Dinesh\\OneDrive\\Desktop\\Microfinance-Fraud-Detection\\data\\processed\\microfinance_fraud_detection_dataset.csv")
print(df.shape)
print(df.head())


(50000, 77)
  branch_id  customer_age  gender marital_status education_level  \
0     BR049            43    Male       Divorced         Diploma   
1     BR174            36    Male        Married         Primary   
2     BR163            45  Female        Married             PhD   
3     BR158            56  Female         Single       Secondary   
4     BR021            35    Male       Divorced       Secondary   

             occupation  annual_income  monthly_expenses  credit_score  \
0               Retired       45591.89          36846.86           563   
1                Farmer       75316.77          50231.18           663   
2  Small Business Owner       53905.76          48578.39           544   
3         Self-Employed       34222.08          28662.02           637   
4     Salaried Employee       89250.89          52902.34           613   

   years_with_institution  ...  application_day_of_week  application_hour  \
0                    6.79  ...                  Tuesday  

In [57]:
X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

In [58]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [59]:
print("Training Features :", X_train.shape)
print("Testing Features  :", X_test.shape)

print()

print("Training Target :", y_train.shape)
print("Testing Target  :", y_test.shape)

Training Features : (40000, 76)
Testing Features  : (10000, 76)

Training Target : (40000,)
Testing Target  : (10000,)


In [60]:
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical:", len(categorical_features))
print("Numerical :", len(numerical_features))

# --- Cardinality check ---
# One-hot encoding every categorical column blows up the feature space when a column
# has many unique values (e.g. branch_id). Columns above CARDINALITY_THRESHOLD unique
# values get frequency-encoded instead (1 column) rather than one-hot encoded (N columns).
CARDINALITY_THRESHOLD = 15

cardinalities = X_train[categorical_features].nunique().sort_values(ascending=False)
print("\nUnique value counts per categorical column:")
print(cardinalities)

high_cardinality_features = cardinalities[cardinalities > CARDINALITY_THRESHOLD].index.tolist()
low_cardinality_features = cardinalities[cardinalities <= CARDINALITY_THRESHOLD].index.tolist()

print(f"\nHigh-cardinality (> {CARDINALITY_THRESHOLD} uniques, will be frequency-encoded):", high_cardinality_features)
print(f"Low-cardinality (<= {CARDINALITY_THRESHOLD} uniques, will be one-hot encoded):", low_cardinality_features)

Categorical: 17
Numerical : 59

Unique value counts per categorical column:
application_date           731
branch_id                  200
state                       20
loan_purpose                 8
occupation                   8
browser_type                 7
application_day_of_week      7
network_type                 7
education_level              7
device_type                  6
application_channel          5
marital_status               4
region                       4
reference_check_result       4
home_ownership               4
gender                       3
sanctions_screening          3
dtype: int64

High-cardinality (> 15 uniques, will be frequency-encoded): ['application_date', 'branch_id', 'state']
Low-cardinality (<= 15 uniques, will be one-hot encoded): ['loan_purpose', 'occupation', 'browser_type', 'application_day_of_week', 'network_type', 'education_level', 'device_type', 'application_channel', 'marital_status', 'region', 'reference_check_result', 'home_ownership', 'ge

C:\Users\Dinesh\AppData\Local\Temp\ipykernel_11512\1050099773.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [61]:
from sklearn.base import BaseEstimator, TransformerMixin

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Replaces each category with how often it appeared in the training data
    (as a proportion, 0-1). Unseen categories at transform time get 0.
    Keeps high-cardinality columns to a single numeric column instead of
    exploding into hundreds of one-hot columns."""

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.freq_maps_ = {
            col: X[col].value_counts(normalize=True) for col in X.columns
        }
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for col in X.columns:
            X[col] = X[col].map(self.freq_maps_[col]).fillna(0.0)
        return X.values.astype(float)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features)


numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

low_card_transformer = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

high_card_transformer = Pipeline(
    steps=[
        ("freq_encoder", FrequencyEncoder()),
        ("scaler", StandardScaler())
    ]
)

transformers = [
    ("num", numeric_transformer, numerical_features),
    ("cat_low", low_card_transformer, low_cardinality_features)
]

if high_cardinality_features:
    transformers.append(("cat_high", high_card_transformer, high_cardinality_features))

preprocessor = ColumnTransformer(transformers=transformers)

In [62]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [63]:
print(X_train_processed.shape)

print(X_test_processed.shape)

(40000, 139)
(10000, 139)


In [64]:
joblib.dump(
    preprocessor,
    "C:\\Users\\Dinesh\\OneDrive\\Desktop\\Microfinance-Fraud-Detection\\models\\preprocessor.pkl "
)

print("Preprocessor Saved Successfully.")

Preprocessor Saved Successfully.


In [65]:
import joblib

joblib.dump(X_train_processed, "C:\\Users\\Dinesh\\OneDrive\\Desktop\\Microfinance-Fraud-Detection\\models\\X_train.pkl")
joblib.dump(X_test_processed, "C:\\Users\\Dinesh\\OneDrive\\Desktop\\Microfinance-Fraud-Detection\\models\\X_test.pkl")

joblib.dump(y_train, "C:\\Users\\Dinesh\\OneDrive\\Desktop\\Microfinance-Fraud-Detection\\models\\y_train.pkl")
joblib.dump(y_test, "C:\\Users\\Dinesh\\OneDrive\\Desktop\\Microfinance-Fraud-Detection\\models\\y_test.pkl")

['C:\\Users\\Dinesh\\OneDrive\\Desktop\\Microfinance-Fraud-Detection\\models\\y_test.pkl']

## Summary

- Loaded cleaned dataset
- Split into training and testing datasets
- Identified numerical and categorical features
- Built a preprocessing pipeline
- Applied scaling and one-hot encoding
- Fitted preprocessing only on the training data
- Transformed the testing data without refitting
- Saved the preprocessing pipeline for reuse